In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# STEP 1: Computing Hashes\n",
    "\n",
    "A **hash** is a short and unique string signature. For example, we may hash the sequence of bytes of a file to obtain an essentially unique code for that file. This allows us to quickly compare two files to see whether they are identical.\n",
    "\n",
    "There are many hash procedures, so we focus on the most important ones, **SHA256** and **MD5**. Note that MD5 is known to exhibit vulnerabilities due to **hash collisions**—instances where two different objects have the same hash and, therefore, should be used with caution.\n",
    "\n",
    "We take an executable file and compute its MD5 and SHA256 hashes."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Download the exe file first\n",
    "!wget https://www.python.org/ftp/python/3.7.2/python-3.7.2-amd64.exe\n",
    "\n",
    "import sys\n",
    "import hashlib\n",
    "\n",
    "filename = \"python-3.7.2-amd64.exe\"\n",
    "BUF_SIZE = 65536\n",
    "\n",
    "md5 = hashlib.md5()\n",
    "sha256 = hashlib.sha256()\n",
    "\n",
    "with open(filename, \"rb\") as f:\n",
    "    while True:\n",
    "        data = f.read(BUF_SIZE)\n",
    "        if not data:\n",
    "            break\n",
    "        md5.update(data)\n",
    "        sha256.update(data)\n",
    "\n",
    "print(\"MD5: {0}\".format(md5.hexdigest()))\n",
    "print(\"SHA256: {0}\".format(sha256.hexdigest()))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "We can verify that our computation is consistent with the hash calculations given by other sources, such as VirusTotal and the official Python website. The MD5 hash is displayed on the Python web page (https://www.python.org/downloads/release/python-372/)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Measuring Similarity Between Strings\n",
    "\n",
    "To check whether two files are **identical**, we utilize standard cryptographic hash functions, such as SHA256 and MD5. If we need to find to what extent two files are **similar**, we use similarity hashing algorithms like `ssdeep`.\n",
    "\n",
    "`ssdeep` can help compare two strings. This can be useful to detect tampering in a text or script and also plagiarism."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!apt-get install -y build-essential libffi-dev python3 python3-dev python3-pip automake autoconf libtool\n",
    "!BUILD_LIB=1 pip install ssdeep"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import ssdeep\n",
    "\n",
    "str1 = \"Lorem ipsum dolor sit amet, consectetur adipiscing elit, sed do eiusmod tempor incididunt ut labore et dolore magna aliqua.\"\n",
    "str2 = \"Lorem ipsum dolor sit amet, consectetur adipiscing elit, sed do eiusmod tempor incididunt ut labore et dolore Magna aliqua.\"\n",
    "str3 = \"Lorem ipsum dolor sit amet, consectetur adipiscing elit, sed do eiusmod tempor incididunt ut labore et dolore aliqua.\"\n",
    "str4 = \"Something completely different from the other strings.\"\n",
    "\n",
    "hash1 = ssdeep.hash(str1)\n",
    "hash2 = ssdeep.hash(str2)\n",
    "hash3 = ssdeep.hash(str3)\n",
    "hash4 = ssdeep.hash(str4)\n",
    "\n",
    "print(f\"Comparing str1 to str1: {ssdeep.compare(hash1, hash1)}\")\n",
    "print(f\"Comparing str1 to str2 (minor change): {ssdeep.compare(hash1, hash2)}\")\n",
    "print(f\"Comparing str1 to str3 (small deletion): {ssdeep.compare(hash1, hash3)}\")\n",
    "print(f\"Comparing str1 to str4 (different): {ssdeep.compare(hash1, hash4)}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Measuring Similarity Between Files\n",
    "\n",
    "We simulate tampering with a file and then use similarity hashing to detect the existence of tampering, as well as measuring the size of the delta. We begin with a vanilla Python executable and then tamper with it by removing the last byte. In real life, a hacker may take a legitimate program and insert malicious code into the sample.\n",
    "\n",
    "We double-check that the tampering was successful and examine its nature using a hexdump. We then run a similarity computation using similarity hashing on the original and tempered file, to observe that a minor alteration took place.\n",
    "\n",
    "Using only standard hashing, we would have no idea how the two files are related, other than to conclude that they are not the same file. Knowing how to compare files allows us to cluster malware and benign files in machine learning algorithms, as well as group them into families."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "import shutil\n",
    "\n",
    "# Ensure the original file exists before copying\n",
    "original_file = \"python-3.7.2-amd64.exe\"\n",
    "tampered_file = \"python-3.7.2-amd64-fake.exe\"\n",
    "\n",
    "if os.path.exists(original_file):\n",
    "    shutil.copyfile(original_file, tampered_file)\n",
    "    # Truncate the file by removing the last byte\n",
    "    os.truncate(tampered_file, os.path.getsize(tampered_file) - 1)\n",
    "\n",
    "    print(\"--- Original File (last 5 lines of hexdump) ---\")\n",
    "    !hexdump -C {original_file} | tail -5\n",
    "    print(\"\\n--- Tampered File (last 5 lines of hexdump) ---\")\n",
    "    !hexdump -C {tampered_file} | tail -5\n",
    "\n",
    "    # Compute and compare similarity hashes\n",
    "    import ssdeep\n",
    "    hash1 = ssdeep.hash_from_file(original_file)\n",
    "    hash2 = ssdeep.hash_from_file(tampered_file)\n",
    "    similarity_score = ssdeep.compare(hash1, hash2)\n",
    "    print(f\"\\nSimilarity score between original and tampered file: {similarity_score}\")\n",
    "else:\n",
    "    print(f\"Error: {original_file} not found. Please run the download cell first.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# STEP 2: YARA\n",
    "\n",
    "*(This section was hidden in the original notebook.)*"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# STEP 3: Examining and Featurizing PE Headers\n",
    "\n",
    "*(This section was hidden in the original notebook.)*"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# STEP 4: N-grams Extraction & Selection\n",
    "\n",
    "## Extracting N-grams\n",
    "\n",
    "In standard quantitative analysis of text, **N-grams** are sequences of N tokens (for example, words or characters).\n",
    "\n",
    "For instance, given the text *The quick brown fox jumped over the lazy dog*, if our tokens are words, then the **1-grams** are `the`, `quick`, `brown`, `fox`, `jumped`, `over`, `the`, `lazy`, and `dog`. The **2-grams** are `the quick`, `quick brown`, `brown fox`, and so on. The **3-grams** are `the quick brown`, `quick brown fox`, `brown fox jumped`, and so on.\n",
    "\n",
    "In our case, we will be using **byte N-grams**. N-grams allow us to model the local statistical properties of the corpus. We use the counts of N-grams to help us predict whether a sample is malicious or benign.\n",
    "\n",
    "First we extract the N-grams from a file."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import collections\n",
    "import nltk\n",
    "# You may need to download the 'punkt' tokenizer models for nltk if you haven't already\n",
    "# nltk.download('punkt')\n",
    "from nltk import ngrams\n",
    "\n",
    "file_to_analyze = \"python-3.7.2-amd64.exe\"\n",
    "\n",
    "def read_file(file_path):\n",
    "    \"\"\"Reads in the binary sequence of a binary file.\"\"\"\n",
    "    with open(file_path, \"rb\") as binary_file:\n",
    "        data = binary_file.read()\n",
    "    return data\n",
    "\n",
    "def byte_sequence_to_Ngrams(byte_sequence, N):\n",
    "    \"\"\"Creates a list of N-grams from a byte sequence.\"\"\"\n",
    "    Ngrams = ngrams(byte_sequence, N)\n",
    "    return list(Ngrams)\n",
    "\n",
    "def binary_file_to_Ngram_counts(file, N):\n",
    "    \"\"\"Takes a binary file and outputs the N-grams counts of its binary sequence.\"\"\"\n",
    "    filebyte_sequence = read_file(file)\n",
    "    file_Ngrams = byte_sequence_to_Ngrams(filebyte_sequence, N)\n",
    "    return collections.Counter(file_Ngrams)\n",
    "\n",
    "# Extract 4-grams from the Python installer\n",
    "extracted_Ngrams = binary_file_to_Ngram_counts(file_to_analyze, 4)\n",
    "\n",
    "print(\"Most common 10 4-grams:\")\n",
    "print(extracted_Ngrams.most_common(10))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Selecting Best N-grams\n",
    "\n",
    "We perform three types of feature selections for N-grams:\n",
    "\n",
    "1.  **Frequency** - selects the most frequent N-grams\n",
    "2.  **Mutual information** — selects the N-grams ranked highest by the mutual information algorithm\n",
    "3.  **Chi-squared** — selects the N-grams ranked highest by the chi-squared algorithm.\n",
    "\n",
    "First, we need a dataset of benign and malicious files. We'll simulate this by creating placeholder directories and files."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# This cell sets up a dummy dataset for demonstration purposes.\n",
    "# In a real scenario, you would have actual malware and benign samples here.\n",
    "import os\n",
    "\n",
    "os.makedirs(\"/content/PE Samples Dataset 2/Benign PE Samples 1\", exist_ok=True)\n",
    "os.makedirs(\"/content/PE Samples Dataset 2/Malicious PE Samples 1\", exist_ok=True)\n",
    "\n",
    "# Create some dummy files\n",
    "with open(\"/content/PE Samples Dataset 2/Benign PE Samples 1/benign1.exe\", \"wb\") as f:\n",
    "    f.write(os.urandom(1024) + b'\\x00\\x00\\x00\\x00' * 100) # Benign files have lots of nulls\n",
    "with open(\"/content/PE Samples Dataset 2/Benign PE Samples 1/benign2.exe\", \"wb\") as f:\n",
    "    f.write(os.urandom(2048) + b'\\x00\\x00' * 300)\n",
    "    \n",
    "with open(\"/content/PE Samples Dataset 2/Malicious PE Samples 1/malicious1.exe\", \"wb\") as f:\n",
    "    f.write(os.urandom(1024) + b'\\x90\\x90\\x90\\x90' * 100) # Malicious files have lots of NOPs\n",
    "with open(\"/content/PE Samples Dataset 2/Malicious PE Samples 1/malicious2.exe\", \"wb\") as f:\n",
    "    f.write(os.urandom(2048) + b'\\x90\\xCC' * 300)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "from os import listdir\n",
    "from os.path import isfile, join\n",
    "\n",
    "base_dir = \"/content/PE Samples Dataset 2/\"\n",
    "directories = [\"Benign PE Samples 1\", \"Malicious PE Samples 1\"]\n",
    "\n",
    "N = 2\n",
    "Ngram_counts_all_files = collections.Counter([])\n",
    "\n",
    "for dataset_path in directories:\n",
    "    current_dataset_path = join(base_dir, dataset_path)\n",
    "    if not os.path.isdir(current_dataset_path):\n",
    "        print(f\"Directory not found: {current_dataset_path}\")\n",
    "        continue\n",
    "        \n",
    "    all_samples = [f for f in listdir(current_dataset_path) if isfile(join(current_dataset_path, f))]\n",
    "    for sample in all_samples:\n",
    "        file_path = join(current_dataset_path, sample)\n",
    "        Ngram_counts_all_files += binary_file_to_Ngram_counts(file_path, N)\n",
    "\n",
    "# Get the top K1 most frequent N-grams from the entire corpus\n",
    "K1 = 1000 \n",
    "K1_most_frequent_Ngrams = Ngram_counts_all_files.most_common(K1)\n",
    "K1_most_frequent_Ngrams_list = [x[0] for x in K1_most_frequent_Ngrams]\n",
    "\n",
    "# Display the top 10 most frequent 2-grams\n",
    "print(\"Top 10 most frequent 2-grams across all files:\")\n",
    "print(K1_most_frequent_Ngrams_list[:10])"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "def featurize_sample(sample, K1_most_frequent_Ngrams_list):\n",
    "    \"\"\"Takes a sample and produces a feature vector.\n",
    "    The features are the counts of the K1 N-grams we've selected.\n",
    "    \"\"\"\n",
    "    K1 = len(K1_most_frequent_Ngrams_list)\n",
    "    feature_vector = K1 * [0]\n",
    "    file_Ngrams = binary_file_to_Ngram_counts(sample, N) # N is a global variable from the previous cell (N=2)\n",
    "    \n",
    "    for i in range(K1):\n",
    "        feature_vector[i] = file_Ngrams[K1_most_frequent_Ngrams_list[i]]\n",
    "    return feature_vector\n",
    "\n",
    "# Prepare dataset for feature selection\n",
    "directories_with_labels = [(\"Benign PE Samples 1\", 0), (\"Malicious PE Samples 1\", 1)]\n",
    "X = []\n",
    "y = []\n",
    "\n",
    "for dataset_path, label in directories_with_labels:\n",
    "    current_dataset_path = join(base_dir, dataset_path)\n",
    "    if not os.path.isdir(current_dataset_path):\n",
    "        print(f\"Directory not found: {current_dataset_path}\")\n",
    "        continue\n",
    "    all_samples = [f for f in listdir(current_dataset_path) if isfile(join(current_dataset_path, f))]\n",
    "    for sample in all_samples:\n",
    "        file_path = join(current_dataset_path, sample)\n",
    "        X.append(featurize_sample(file_path, K1_most_frequent_Ngrams_list))\n",
    "        y.append(label)\n",
    "\n",
    "import numpy as np\n",
    "X = np.asarray(X)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from sklearn.feature_selection import SelectKBest, mutual_info_classif, chi2\n",
    "\n",
    "K2 = 10 # We want to select the top 10 features\n",
    "\n",
    "# 1. Frequency based selection (already done by how we constructed X)\n",
    "X_top_K2_freq = X[:, :K2]\n",
    "print(f\"Shape of top {K2} frequency features: {X_top_K2_freq.shape}\")\n",
    "\n",
    "# 2. Mutual Information based selection\n",
    "mi_selector = SelectKBest(mutual_info_classif, k=K2)\n",
    "X_top_K2_mi = mi_selector.fit_transform(X, y)\n",
    "print(f\"Shape of top {K2} mutual information features: {X_top_K2_mi.shape}\")\n",
    "\n",
    "# 3. Chi-squared based selection\n",
    "chi2_selector = SelectKBest(chi2, k=K2)\n",
    "X_top_K2_ch2 = chi2_selector.fit_transform(X, y)\n",
    "print(f\"Shape of top {K2} Chi2 features: {X_top_K2_ch2.shape}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# STEP 5: Static Malware Detector\n",
    "\n",
    "We build a malware detector taking both features extracted from the PE header as well as features derived from N-grams.\n",
    "\n",
    "**Static analysis** involves examining a sample without executing it. The amount of information that can be obtained this way is large, ranging from something as simple as the name of the file to the more complex, such as specialized YARA signatures. We will be covering a selection of the large variety of features you could obtain by statically analyzing a sample. Despite its power and convenience, static analysis is no silver bullet, mainly because software can be obfuscated."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Data Preparation and Splitting\n",
    "\n",
    "First, we collect all file paths and their corresponding labels (0 for benign, 1 for malicious)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "from os import listdir\n",
    "from os.path import isfile, join\n",
    "\n",
    "directories_with_labels = [(\"Benign PE Samples 1\", 0), (\"Malicious PE Samples 1\", 1)]\n",
    "list_of_samples = []\n",
    "labels = []\n",
    "\n",
    "base_dir = \"/content/PE Samples Dataset 2/\"\n",
    "\n",
    "for dataset_path, label in directories_with_labels:\n",
    "    current_dataset_path = join(base_dir, dataset_path)\n",
    "    if not os.path.isdir(current_dataset_path):\n",
    "        print(f\"Directory not found: {current_dataset_path}\")\n",
    "        continue\n",
    "    samples = [f for f in listdir(current_dataset_path)]\n",
    "    for sample in samples:\n",
    "        file_path = os.path.join(current_dataset_path, sample)\n",
    "        list_of_samples.append(file_path)\n",
    "        labels.append(label)\n",
    "        \n",
    "print(f\"Total samples: {len(list_of_samples)}\")\n",
    "print(f\"Total labels: {len(labels)}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "Because our dataset might be imbalanced, it makes sense to use a **stratified train-test split**. In a stratified train-test split, a train-test split is created in which the proportion of each class is the same in the training set, testing set, and original set. This ensures that there is no possibility that our training set, for example, will consist of only one class due to a chance event."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from sklearn.model_selection import train_test_split\n",
    "\n",
    "samples_train, samples_test, labels_train, labels_test = train_test_split(\n",
    "    list_of_samples, labels, test_size=0.3, stratify=labels, random_state=11\n",
    ")\n",
    "\n",
    "print(f\"Training samples: {len(samples_train)}\")\n",
    "print(f\"Testing samples: {len(samples_test)}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Feature Extraction\n",
    "\n",
    "Now we define all the functions needed to extract features from a Portable Executable (PE) file. This includes:\n",
    "1.  **N-gram counts**: The frequency of the top N-grams found in the training set.\n",
    "2.  **Imports**: The list of imported DLLs.\n",
    "3.  **Section Names**: The names of the sections in the PE header.\n",
    "4.  **Number of Sections**: The total count of sections."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!pip install pefile\n",
    "\n",
    "import collections\n",
    "import numpy as np\n",
    "import pefile\n",
    "from nltk import ngrams\n",
    "\n",
    "def read_file(file_path):\n",
    "    \"\"\"Reads in the binary sequence of a binary file.\"\"\"\n",
    "    with open(file_path, \"rb\") as binary_file:\n",
    "        data = binary_file.read()\n",
    "    return data\n",
    "\n",
    "def byte_sequence_to_Ngrams(byte_sequence, N):\n",
    "    \"\"\"Creates a list of N-grams from a byte sequence.\"\"\"\n",
    "    Ngrams = ngrams(byte_sequence, N)\n",
    "    return list(Ngrams)\n",
    "\n",
    "def binary_file_to_Ngram_counts(file, N):\n",
    "    \"\"\"Takes a binary file and outputs the N-grams counts of its binary sequence.\"\"\"\n",
    "    filebyte_sequence = read_file(file)\n",
    "    file_Ngrams = byte_sequence_to_Ngrams(filebyte_sequence, N)\n",
    "    return collections.Counter(file_Ngrams)\n",
    "\n",
    "def get_NGram_features_from_sample(sample, K1_most_frequent_Ngrams_list):\n",
    "    \"\"\"Takes a sample and produces a feature vector from N-gram counts.\"\"\"\n",
    "    K1 = len(K1_most_frequent_Ngrams_list)\n",
    "    feature_vector = K1 * [0]\n",
    "    file_Ngrams = binary_file_to_Ngram_counts(sample, N) # N is global\n",
    "    for i in range(K1):\n",
    "        feature_vector[i] = file_Ngrams[K1_most_frequent_Ngrams_list[i]]\n",
    "    return feature_vector\n",
    "\n",
    "def preprocess_imports(list_of_DLLs):\n",
    "    \"\"\"Normalize the naming of the imports of a PE file.\"\"\"\n",
    "    temp = [x.decode().split(\".\")[0].lower() for x in list_of_DLLs]\n",
    "    return \" \".join(temp)\n",
    "\n",
    "def get_imports(pe):\n",
    "    \"\"\"Get a list of the imports of a PE file.\"\"\"\n",
    "    list_of_imports = []\n",
    "    if hasattr(pe, 'DIRECTORY_ENTRY_IMPORT'):\n",
    "        for entry in pe.DIRECTORY_ENTRY_IMPORT:\n",
    "            list_of_imports.append(entry.dll)\n",
    "    return preprocess_imports(list_of_imports)\n",
    "\n",
    "def get_section_names(pe):\n",
    "    \"\"\"Gets a list of section names from a PE file.\"\"\"\n",
    "    list_of_section_names = []\n",
    "    for sec in pe.sections:\n",
    "        normalized_name = sec.Name.decode().replace(\"\\x00\", \"\").lower()\n",
    "        list_of_section_names.append(normalized_name)\n",
    "    return \" \".join(list_of_section_names)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "First, we determine the most common N-grams from the **training set only**. This is crucial to avoid data leakage from the test set into our feature selection process."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "N = 2\n",
    "K1 = 100 # Number of N-gram features to use\n",
    "\n",
    "Ngram_counts_all = collections.Counter([])\n",
    "for sample in samples_train:\n",
    "    Ngram_counts_all += binary_file_to_Ngram_counts(sample, N)\n",
    "\n",
    "K1_most_frequent_Ngrams = Ngram_counts_all.most_common(K1)\n",
    "K1_most_frequent_Ngrams_list = [x[0] for x in K1_most_frequent_Ngrams]"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "Now, we extract the N-gram counts, section names, imports, and number of sections of each sample in our **training set**, and skip over samples whose PE header cannot be parsed."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "imports_corpus_train = []\n",
    "num_sections_train = []\n",
    "section_names_train = []\n",
    "Ngram_features_list_train = []\n",
    "y_train = []\n",
    "\n",
    "for i in range(len(samples_train)):\n",
    "    sample = samples_train[i]\n",
    "    try:\n",
    "        NGram_features = get_NGram_features_from_sample(\n",
    "            sample, K1_most_frequent_Ngrams_list\n",
    "        )\n",
    "        pe = pefile.PE(sample, fast_load=True) # Use fast_load for efficiency\n",
    "        imports = get_imports(pe)\n",
    "        n_sections = len(pe.sections)\n",
    "        sec_names = get_section_names(pe)\n",
    "        pe.close()\n",
    "\n",
    "        imports_corpus_train.append(imports)\n",
    "        num_sections_train.append(n_sections)\n",
    "        section_names_train.append(sec_names)\n",
    "        Ngram_features_list_train.append(NGram_features)\n",
    "        y_train.append(labels_train[i])\n",
    "    except Exception as e:\n",
    "        print(f\"Error processing {sample}: {e}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Vectorizing Text Features\n",
    "\n",
    "We use a `HashingVectorizer` followed by `TfidfTransformer` to convert the imports and section names, both of which are text features, into a numerical form. This is a common and efficient technique for handling text data in machine learning."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer\n",
    "from sklearn.pipeline import Pipeline\n",
    "\n",
    "imports_featurizer = Pipeline(\n",
    "    [\n",
    "        (\"vect\", HashingVectorizer(input=\"content\", ngram_range=(1, 2))),\n",
    "        (\"tfidf\", TfidfTransformer(use_idf=True,)),\n",
    "    ]\n",
    ")\n",
    "\n",
    "section_names_featurizer = Pipeline(\n",
    "    [\n",
    "        (\"vect\", HashingVectorizer(input=\"content\", ngram_range=(1, 2))),\n",
    "        (\"tfidf\", TfidfTransformer(use_idf=True,)),\n",
    "    ]\n",
    ")\n",
    "\n",
    "imports_corpus_train_transformed = imports_featurizer.fit_transform(\n",
    "    imports_corpus_train\n",
    ")\n",
    "section_names_train_transformed = section_names_featurizer.fit_transform(\n",
    "    section_names_train\n",
    ")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Assembling the Final Feature Matrix\n",
    "\n",
    "We combine the vectorized features into a single array. Having obtained all these different features, we are now ready to combine them, which we do using `scipy.sparse.hstack`, to merge the different features into one large sparse `scipy` array. This is memory-efficient."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from scipy.sparse import hstack, csr_matrix\n",
    "\n",
    "X_train = hstack(\n",
    "    [\n",
    "        csr_matrix(Ngram_features_list_train),\n",
    "        imports_corpus_train_transformed,\n",
    "        section_names_train_transformed,\n",
    "        csr_matrix(num_sections_train).transpose(),\n",
    "    ]\n",
    ")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Training the Classifier\n",
    "\n",
    "We train a Random Forest classifier on the training set and print out its score."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from sklearn.ensemble import RandomForestClassifier\n",
    "\n",
    "clf = RandomForestClassifier(n_estimators=100, random_state=42)\n",
    "clf = clf.fit(X_train, y_train)\n",
    "\n",
    "training_score = clf.score(X_train, y_train)\n",
    "print(f\"Accuracy on the training set: {training_score:.4f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Evaluating the Classifier on the Test Set\n",
    "\n",
    "We collect the features of the testing set, just as we did for the training set."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "imports_corpus_test = []\n",
    "num_sections_test = []\n",
    "section_names_test = []\n",
    "Ngram_features_list_test = []\n",
    "y_test = []\n",
    "\n",
    "for i in range(len(samples_test)):\n",
    "    file = samples_test[i]\n",
    "    try:\n",
    "        # The sample variable was not defined in the original code for the test loop.\n",
    "        # Corrected to use 'file' instead of 'sample'.\n",
    "        NGram_features = get_NGram_features_from_sample(\n",
    "            file, K1_most_frequent_Ngrams_list \n",
    "        )\n",
    "        pe = pefile.PE(file, fast_load=True)\n",
    "        imports = get_imports(pe)\n",
    "        n_sections = len(pe.sections)\n",
    "        sec_names = get_section_names(pe)\n",
    "        pe.close()\n",
    "\n",
    "        imports_corpus_test.append(imports)\n",
    "        num_sections_test.append(n_sections)\n",
    "        section_names_test.append(sec_names)\n",
    "        Ngram_features_list_test.append(NGram_features)\n",
    "        y_test.append(labels_test[i])\n",
    "    except Exception as e:\n",
    "        # In the original code, 'sample' was used here, which would cause an error.\n",
    "        # Corrected to use 'file'.\n",
    "        print(f\"Error processing {file}: {e}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "We apply the **previously trained** transformers to vectorize the text features and then test our classifier on the resulting test set. It's important to use `transform` here, not `fit_transform`, to ensure the test data is processed in the exact same way as the training data."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "imports_corpus_test_transformed = imports_featurizer.transform(imports_corpus_test)\n",
    "section_names_test_transformed = section_names_featurizer.transform(section_names_test)\n",
    "\n",
    "X_test = hstack(\n",
    "    [\n",
    "        csr_matrix(Ngram_features_list_test),\n",
    "        imports_corpus_test_transformed,\n",
    "        section_names_test_transformed,\n",
    "        csr_matrix(num_sections_test).transpose(),\n",
    "    ]\n",
    ")\n",
    "\n",
    "testing_score = clf.score(X_test, y_test)\n",
    "print(f\"Accuracy on the testing set: {testing_score:.4f}\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.10.6"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}